# EX12 — Metoda SOR (Successive Over-Relaxation)

Metoda SOR jest uogólnieniem metody Gaussa-Seidla z dodatnim parametrem relaksacji $\omega$. Przy $\omega=1$ metoda SOR sprowadza się do metody Gaussa-Seidla. Dla $\omega > 1$ mówimy o nadrelaksacji (over-relaxation), co może przyspieszyć zbieżność.

Rzutowana iteracja ma postać:

$$x^{(k+1)} = (D + \omega L)^{-1}\bigl[(1-\omega)D - \omega U\bigr]x^{(k)} + \omega(D + \omega L)^{-1}b$$

gdzie $A = D + L + U$ to rozkład macierzy na część diagonalną, dolnotrójkątną i górną. Macierz iteracyjna SOR wynosi:

$$M_{\omega} = (D + \omega L)^{-1}\bigl[(1-\omega)D - \omega U\bigr]$$

Stosujemy metodę na zmodyfikowanej macierzy Wilkinsona $A + 10I$ z wektorem $b = A\cdot\mathbf{1}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def wilkinson(n):
    A = np.diag(np.arange(n, 0, -1), 0) + np.diag(-np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1)
    return A.astype(np.float64)

A = wilkinson(5) + 10 * np.eye(5)
b = A.sum(axis=1)
x_dokl = np.ones(5)

D = np.diag(np.diag(A))
L_mat = np.tril(A) - D   # lower triangular (negative convention)
U_mat = np.triu(A) - D   # upper triangular

x0 = np.random.rand(5)
delta0 = 1e-8
i_max = 50

for omega in [1.8, 1.7, 1.6]:
    inv_DwL = np.linalg.inv(D + omega * L_mat)
    M_omega = inv_DwL @ ((1 - omega) * D - omega * U_mat)
    w_omega = omega * inv_DwL @ b

    spectrumM = np.max(np.abs(np.linalg.eigvals(M_omega)))
    print(f"ω={omega:.1f} — Promień spektralny: {spectrumM:.2f}")

    x = x0.copy()
    err = [np.linalg.norm(x - x_dokl) / np.linalg.norm(x_dokl)]
    n_iter = [0]
    i = 1
    while err[-1] >= delta0 and i < i_max:
        x = M_omega @ x + w_omega
        err.append(np.linalg.norm(x - x_dokl) / np.linalg.norm(x_dokl))
        n_iter.append(i)
        i += 1

    plt.semilogy(n_iter, err, label=f'ω={omega:.1f} (ρ={spectrumM:.2f})')

plt.legend()
plt.xlabel('# iteracji')
plt.ylabel('Estymata błędu')
plt.grid(True)
plt.show()

## ĆWICZENIE 2

**Warunki zbieżności metody SOR:**

1. Metoda SOR zbiega dla dowolnego $x^{(0)}$ wtedy i tylko wtedy, gdy promień spektralny macierzy iteracyjnej spełnia:
   $$\rho(M_{\omega}) < 1$$

2. Koniecznym warunkiem zbieżności jest $\omega \in (0,\,2)$. Dla $\omega \ge 2$ i $\omega \le 0$ metoda zawsze rozbiega.

3. Dla macierzy symetrycznej dodatnio określonej ($A = A^T \succ 0$) każdy $\omega \in (0,\,2)$ zapewnia zbieżność.

**Optymalny parametr relaksacji:**

Jeśli $A$ jest macierzą o dominującej diagonali i $\lambda_{\max}$ oznacza największą wartość własną macierzy $D^{-1}(L+U)$, to optymalny parametr relaksacji wyznaczamy ze wzoru:

$$\omega_{\mathrm{opt}} = \frac{2}{1 + \sqrt{1 - \lambda_{\max}^2}}$$

Dla tego wyboru $\omega$ promień spektralny macierzy iteracyjnej osiąga minimum:

$$\rho(M_{\omega_{\mathrm{opt}}}) = \omega_{\mathrm{opt}} - 1$$

**Zadanie:** Dla macierzy z Ćwiczenia 1 oblicz $\lambda_{\max}$ macierzy $D^{-1}(L+U)$ i porównaj $\omega_{\mathrm{opt}}$ obliczone ze wzoru z wartościami $\omega \in \{1.6, 1.7, 1.8\}$. Który parametr daje najmniejszy promień spektralny?

## Wykorzystywane funkcje

| Funkcja | Opis |
|---------|------|
| `np.diag(v, k)` | Tworzy macierz diagonalną z wektora `v` na $k$-tej przekątnej |
| `np.tril(A)` | Dolna część macierzy $A$ |
| `np.triu(A)` | Górną część macierzy $A$ |
| `np.linalg.inv(A)` | Odwraca macierz kwadratową |
| `np.linalg.eigvals(A)` | Wartości własne macierzy $A$ |
| `np.linalg.norm(x)` | Norma euklidesowa wektora |
| `plt.semilogy(x, y)` | Wykres z logarytmiczną osią $y$ |
| `wilkinson(n)` | Macierz Wilkinsona rozmiaru $n \times n$ |